# Day 36 — Practice ANSWERS
### Honest Scores — train/test split, cross-validation, pipelines

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### Four questions to keep asking

- Is this score measured on rows the model **studied**, or rows it **never saw**?
- What is the **baseline**? A score without one means nothing.
- Was this **one split**, or an average of several?
- Did any preparation step look at **all** the rows before the folds were made?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     LeaveOneOut, cross_val_score, cross_validate,
                                     cross_val_predict)
from sklearn.metrics import accuracy_score, confusion_matrix

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

No code needed for most of these. Work them out, then run the cell to check.

### A1 · Fold arithmetic

You have **250 rows** and you run `StratifiedKFold(n_splits=5)`.

1. How many rows are held back in each fold?
2. How many rows does the model study in each fold?
3. How many times is each individual row **held back**?
4. How many times is each individual row **studied**?
5. How many models get fitted in total?

1. **50 rows** held back — 250 / 5
2. **200 rows** studied — the other four fifths
3. **Exactly once.** That is the whole design: every row gets one turn in the exam.
4. **Four times** — it appears in the training set of every fold except its own
5. **Five models.** All five are then thrown away; what you keep is the five scores.

**The point of 3 and 4 together:** cross-validation wastes nothing. Every row contributes
to training *and* gets scored exactly once, which is why it uses a small dataset so much
better than a single split does.

### A2 · How much is one row worth?

You hold back 30% of a dataset and report accuracy on it.

How much does **one row** change the reported accuracy when the dataset has
100 rows? 400 rows? 5,000 rows?

In [2]:
for n in [100, 400, 5000]:
    held = int(n * 0.3)
    print(f"{n:>6} rows  ->  {held:>5} held back  ->  one row is worth {1/held:.4f}")

   100 rows  ->     30 held back  ->  one row is worth 0.0333
   400 rows  ->    120 held back  ->  one row is worth 0.0083
  5000 rows  ->   1500 held back  ->  one row is worth 0.0007


```
   100 rows  ->     30 held back  ->  one row is worth 0.0333
   400 rows  ->    120 held back  ->  one row is worth 0.0083
  5000 rows  ->   1500 held back  ->  one row is worth 0.0007
```

**On 100 rows, two students' difference is nearly 7 percentage points.** That is the whole
reason a single split on a small dataset swings so hard — and the reason the same
experiment on 2,000 rows in the lecture had a spread of 4.7 points instead of 16.7.

A useful habit: before quoting an accuracy, work out what one row is worth. If it is
worth more than the difference you are excited about, you have not found anything.

### A3 · Read a cross-validation result

Two models on the same data:

| | fold scores |
|---|---|
| **Model P** | 0.81, 0.79, 0.82, 0.80, 0.78 |
| **Model Q** | 0.95, 0.62, 0.88, 0.71, 0.84 |

Work out each one's mean and standard deviation. Which would you deploy, and why?

In [3]:
P = np.array([0.81, 0.79, 0.82, 0.80, 0.78])
Q = np.array([0.95, 0.62, 0.88, 0.71, 0.84])
for name, a in [("P", P), ("Q", Q)]:
    print(f"model {name}:  mean {a.mean():.4f}   sd {a.std(ddof=1):.4f}   "
          f"range {a.min():.2f} to {a.max():.2f}")

model P:  mean 0.8000   sd 0.0158   range 0.78 to 0.82
model Q:  mean 0.8000   sd 0.1332   range 0.62 to 0.95


```
model P:  mean 0.8000   sd 0.0158   range 0.78 to 0.82
model Q:  mean 0.8000   sd 0.1332   range 0.62 to 0.95
```

**Identical means. Completely different results.**

**Deploy P.** Not because it scores better — it does not — but because you know what you
are getting. P will land near 0.80 on new data. Q might land at 0.62.

**And Q is telling you something else as well.** A spread that wide on five folds of the
same dataset usually means one of three things:

- the dataset is too small for the model's flexibility (overfitting, fold by fold)
- there is a group structure you have not accounted for — one fold got all the hard cases
- the folds were not stratified, so the class mix moved

In every one of those, *investigate before deploying*. The spread is not noise to be
averaged away. It is the warning.

**This is exactly what a single split hides.** Had you run one split on model Q and
happened to get 0.95, you would have shipped it.

### A4 · Which of these leak?

For each step, say whether doing it **before** splitting leaks, and why.

1. `StandardScaler` on all the rows
2. Dropping a column that is 95% empty
3. `SelectKBest(f_classif, k=10)` on all the rows
4. Filling missing ages with the median age of all the rows
5. Trying `max_depth` from 1 to 20 and keeping whichever scored best
6. `OneHotEncoder` on all the rows

| | step | leaks? | why |
|---|---|---|---|
| 1 | `StandardScaler` | **barely** | a mean and an sd barely move when you add 20% more rows — measured at **+0.0000** in the lecture |
| 2 | dropping a 95%-empty column | **no** | that decision uses only the columns, never `y` |
| 3 | `SelectKBest` | **YES, badly** | `f_classif` scores each column *against `y`* — this was the +0.2750 disaster |
| 4 | filling with the median | **barely** | the median never looks at `y`; measured at −0.0067 |
| 5 | trying 20 depths, keeping the best | **YES** | "best of 20" is a choice made using the scores, so it is a choice made using `y` |
| 6 | `OneHotEncoder` | **no** (almost) | the category list never looks at `y`. *One caveat:* if a category appears only in the hidden rows, encoding up front quietly tells you it exists |

### The rule

> **A step that never looks at `y` leaks almost nothing.
> A step that makes a choice using `y` can leak everything.**

**Number 5 is the one people miss.** It does not feel like data preparation at all — it
feels like modelling. But picking the winner of twenty attempts is exactly as much a
choice made from the answers as `SelectKBest` is, and it needs the same protection: a
final test set that nothing has touched.

### A5 · Why is leave-one-out's ± meaningless?

A model run with `LeaveOneOut()` on 200 rows reports **mean 0.8500, sd 0.3580**.

Explain where the 0.3579 comes from, and what it would be if the model had got
170 of the 200 right in a completely different way.

In [4]:
n = 200
for correct in [170, 170]:
    folds = np.array([1.0]*correct + [0.0]*(n-correct))
    print(f"{correct} right of {n}:  mean {folds.mean():.4f}   sd {folds.std(ddof=1):.4f}")

p = 0.85
print(f"\nformula for ANY list of {n} zeros/ones with mean {p}: "
      f"{np.sqrt(p*(1-p)*n/(n-1)):.4f}")

170 right of 200:  mean 0.8500   sd 0.3580
170 right of 200:  mean 0.8500   sd 0.3580

formula for ANY list of 200 zeros/ones with mean 0.85: 0.3580


Each leave-one-out fold trains on 199 rows and tests on **one**. So each fold's "accuracy"
is either **1.0** or **0.0** — there is no third possibility.

That makes the list of fold scores just 170 ones and 30 zeros, and the standard deviation
of such a list is **completely determined by the mean**:

```
sd = sqrt( p(1-p) · n/(n-1) ) = sqrt(0.85 × 0.15 × 200/199) = 0.3580
```

**Any model that got 170 of 200 right reports exactly 0.3580**, however stable or
unstable it actually is. The number measures nothing about the model. It measures the
fact that 0 and 1 are far apart.

**What to do:** with leave-one-out, quote the mean and say nothing about the spread. And
prefer 5- or 10-fold for anything above about 50 rows.

### A6 · The sorted file

A hospital exports 500 patient records, sorted by outcome: all the survivors first, then
all the deaths. 20% died.

You run `KFold(n_splits=5)` with no other arguments.

1. How many rows in each fold?
2. What is the death rate inside fold 1? Inside fold 5?
3. What will a model that predicts "survived" for everybody score on each fold?

1. **100 rows per fold.**

2. The file is 400 survivors followed by 100 deaths, and `KFold` with no shuffle takes
   consecutive blocks:

| fold | rows | contents | death rate |
|---|---|---|---|
| 1 | 1–100 | all survivors | **0.00** |
| 2 | 101–200 | all survivors | **0.00** |
| 3 | 201–300 | all survivors | **0.00** |
| 4 | 301–400 | all survivors | **0.00** |
| 5 | 401–500 | all deaths | **1.00** |

3. "Always survived" scores **1.0000** on folds 1–4 and **0.0000** on fold 5 — a reported
   mean of **0.8000**.

### Why that 0.8000 is a trap

It is the correct overall accuracy *by coincidence* — 80% of patients did survive. But
look at what produced it: four perfect folds and one catastrophic one. No model was
evaluated meaningfully even once.

Worse, a **real** model would do badly here: in folds 1–4 it has studied a training set
containing every single death, and in fold 5 it has studied one containing none at all.
Both are nothing like the data it will meet in service.

**The fix is two keyword arguments**: `StratifiedKFold(n_splits=5, shuffle=True)`.

### A7 · Does this need a final test set?

For each, say whether cross-validation alone is honest, or whether you also need a test
set that nothing has touched.

1. Fit one logistic regression. Report its 5-fold CV score.
2. Fit logistic regression, a decision tree and kNN. Report the best CV score of the three.
3. Try `C` = 0.01, 0.1, 1, 10, 100. Report the best CV score.
4. Fit one model, report its CV score, and *then* decide it is good enough to deploy.

| | cross-validation alone? | why |
|---|---|---|
| 1 | **yes, honest** | you made no choice using the scores |
| 2 | **no — you need a test set** | "best of three" is a choice made using `y` |
| 3 | **no — you need a test set** | "best of five" is the same thing |
| 4 | **yes, honest** | deciding to deploy is not choosing between models; you did not pick this model *because* it scored well against alternatives |

### Why 2 and 3 need one

Run five identical coins through cross-validation and the best of the five will look
better than a coin. **Best-of-N is not an average of N** — the winner is high partly on
merit and partly on luck, and the more things you try, the more of it is luck.

Only rows that played no part in the choosing can tell you how much of each.

```
all your rows
   -> 80% working set   cross-validate in here as much as you like
   -> 20% final test    opened once, at the very end
```

Case 4 is worth thinking about carefully. The distinction is not "did I make a decision"
— it is "did I use the scores to **select among options**". Deciding to ship the only
model you built uses no comparison, so it adds no optimism.

### A8 · Which splitter?

Pick the right cross-validator, and say in one line why.

1. 900 loan applications, 38% approved, rows in no particular order
2. House prices over time; you want to predict next quarter
3. 4,000 X-rays from 800 patients (several images each), predicting a fracture
4. 200 emails, 4% spam

**1 · `StratifiedKFold(5, shuffle=True)`** — classification, so keep each fold's approval
rate near 38%. Shuffle regardless of what you were told about the row order; it costs
nothing and you cannot always see a sort.

**2 · `TimeSeriesSplit`** — always train on the past and test on the future. Shuffling
time-ordered data lets the model learn from next quarter to predict last quarter, which
it will never be able to do in service. This is one of the easiest ways to produce a
spectacular score that collapses on deployment.

**3 · `GroupKFold`, grouped by patient** — this is the important one. With plain
`StratifiedKFold`, two X-rays of the *same patient's* chest land on opposite sides of the
split. The model studies that patient and is then tested on them. "Rows it never saw" has
to mean genuinely never saw, and here that means **patients**, not images.

**4 · `StratifiedKFold`, and think hard about the whole design** — 4% of 200 is **8 spam
emails**. Five folds gives you 1 or 2 per fold. Stratifying is essential just to make sure
no fold has zero. But honestly: accuracy is useless here (always-"not spam" scores 96%),
you should be reporting recall, and 8 positive cases is barely enough to measure anything.
The right answer to this one includes *"get more spam examples"*.

---
# Part B — Find the mistake

Four short programs. Each runs cleanly and reports a number that is wrong.

In [5]:
# ---------------------------------------------------------------
# 400 clinic appointments. B and C both use `clinic`.
# ---------------------------------------------------------------
r = np.random.default_rng(3650)
N = 400

age               = r.normal(34, 14, N).clip(5, 85).round(0)
lead_time_days    = r.uniform(0, 60, N).round(0)
previous_no_shows = r.poisson(0.7, N).clip(0, 6).astype(float)
distance_km       = r.normal(9, 6, N).clip(0.5, 45).round(1)
reminder          = r.choice(["none", "sms", "call"], N, p=[.35, .45, .20])

rb = np.where(reminder == "none", 0.45, np.where(reminder == "sms", 0.0, -0.40))
z = (-4.1 + 3.5*(0.022*lead_time_days + 0.55*previous_no_shows
                 - 0.012*age + 0.020*distance_km + rb) + r.normal(0, 0.45, N))
no_show = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

clinic = pd.DataFrame({"age": age, "lead_time_days": lead_time_days,
                       "previous_no_shows": previous_no_shows,
                       "distance_km": distance_km, "reminder": reminder,
                       "no_show": no_show})
clinic.loc[r.uniform(0, 1, N) < 0.10, "distance_km"] = np.nan   # some addresses missing

NUM = ["age", "lead_time_days", "previous_no_shows", "distance_km"]
CAT = ["reminder"]
print("rows:", len(clinic), "  no-show rate:", round(clinic.no_show.mean(), 4))
print("missing distances:", int(clinic.distance_km.isna().sum()))
clinic.head()

rows: 400   no-show rate: 0.395
missing distances: 28


,age,lead_time_days,previous_no_shows,distance_km,reminder,no_show
0,55.0,54.0,1.0,NaN,call,0
1,34.0,7.0,1.0,0.5,sms,0
2,46.0,25.0,1.0,0.5,sms,0
3,44.0,5.0,1.0,5.4,none,1
4,33.0,9.0,1.0,9.8,sms,0


In [6]:
# a filled, numeric copy for the simpler questions
flat = pd.get_dummies(clinic[NUM + CAT], columns=CAT, drop_first=True).astype(float)
flat["distance_km"] = flat["distance_km"].fillna(flat["distance_km"].median())
target = clinic.no_show
print(list(flat.columns))

['age', 'lead_time_days', 'previous_no_shows', 'distance_km', 'reminder_none', 'reminder_sms']


### B1 · "Our model is 100% accurate"

In [7]:
X_train, X_hidden, y_train, y_hidden = train_test_split(
    flat, target, test_size=0.3, random_state=0, stratify=target)

tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

print(f"accuracy: {tree.score(X_train, y_train):.4f}")
print()
print("REPORT: 'The no-show model is 100% accurate. Ready to deploy.'")

accuracy: 1.0000

REPORT: 'The no-show model is 100% accurate. Ready to deploy.'


**The problem:** the score is measured on `X_train` — **the rows the tree studied.**

A decision tree with no depth limit grows one branch per training row. Asked about those
same rows it looks them up and is never wrong. The 1.0000 says nothing about the model
and everything about its flexibility.

**The fix:** score `X_hidden`, the rows it has never seen.

In [8]:
print(f"{'model':<28}{'studied':>10}{'hidden':>10}{'gap':>10}")
print("-" * 58)
for nm, m in [("DecisionTree (no limit)", DecisionTreeClassifier(random_state=0)),
              ("DecisionTree max_depth=3", DecisionTreeClassifier(max_depth=3, random_state=0)),
              ("LogisticRegression", LogisticRegression(max_iter=2000))]:
    m.fit(X_train, y_train)
    a, b = m.score(X_train, y_train), m.score(X_hidden, y_hidden)
    print(f"{nm:<28}{a:>10.4f}{b:>10.4f}{a-b:>+10.4f}")
print("-" * 58)
print(f"{'guessing they all turn up':<28}{'':>10}{1-y_hidden.mean():>10.4f}")

model                          studied    hidden       gap
----------------------------------------------------------
DecisionTree (no limit)         1.0000    0.7250   +0.2750
DecisionTree max_depth=3        0.7607    0.7333   +0.0274
LogisticRegression              0.8179    0.8167   +0.0012
----------------------------------------------------------
guessing they all turn up                 0.6083


```
model                          studied    hidden       gap
----------------------------------------------------------
DecisionTree (no limit)         1.0000    0.7250   +0.2750
DecisionTree max_depth=3        0.7607    0.7333   +0.0274
LogisticRegression              0.8179    0.8167   +0.0012
----------------------------------------------------------
guessing they all turn up                 0.6083
```

**The "100% accurate" tree is the worst of the three on rows it has never seen** — 0.7250,
beaten even by the depth-3 tree that scored a modest 0.7607 on its own rows.

The logistic regression scores **0.8167** with a gap of **0.0012**. That is what a model
that has actually learned something looks like: the two columns agree.

**A gap is not automatically fatal** — some gap is normal. A gap of 0.2750 next to a
perfect training score is memorisation.

### B2 · The seed that was tried until it worked

In [9]:
X_train, X_hidden, y_train, y_hidden = train_test_split(
    flat, target, test_size=0.3, random_state=12, stratify=target)

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
print(f"accuracy on held-back rows: {model.score(X_hidden, y_hidden):.4f}")
print()
print("REPORT: 'The model reaches 86.7% on unseen appointments.'")

accuracy on held-back rows: 0.8667

REPORT: 'The model reaches 86.7% on unseen appointments.'


**The problem:** `random_state=12` is a suspiciously specific choice, and the number it
produces is a single draw from a lottery. Nothing here is technically *wrong* — the rows
really were held back — but one split on 400 rows is one measurement, and quoting the
best one you found is how optimistic results get published without anyone lying.

**The fix:** cross-validate, and report the mean with its spread.

In [10]:
accs = []
for seed in range(20):
    a, b, c, d = train_test_split(flat, target, test_size=0.3,
                                  random_state=seed, stratify=target)
    accs.append(LogisticRegression(max_iter=2000).fit(a, c).score(b, d))
accs = np.array(accs)

print("twenty seeds:")
for i in range(0, 20, 5):
    print("   " + "  ".join(f"{v:.4f}" for v in accs[i:i+5]))
print()
print(f"lowest  {accs.min():.4f}  (seed {accs.argmin()})")
print(f"highest {accs.max():.4f}  (seed {accs.argmax()})")
print(f"spread  {accs.max()-accs.min():.4f}")
print(f"mean    {accs.mean():.4f}   sd {accs.std(ddof=1):.4f}")

twenty seeds:
   0.8167  0.7750  0.8500  0.8167  0.8167
   0.8000  0.8000  0.8167  0.7917  0.7750
   0.7500  0.7917  0.8667  0.8333  0.8417
   0.8333  0.8500  0.7750  0.8250  0.7833

lowest  0.7500  (seed 10)
highest 0.8667  (seed 12)
spread  0.1167
mean    0.8104   sd 0.0306


```
twenty seeds:
   0.8083  0.8167  0.8083  0.8083  0.8250
   0.8000  0.8250  0.8000  0.7833  0.8333
   0.7500  0.8250  0.8667  0.8250  0.8167
   0.8083  0.7917  0.8167  0.8000  0.7833

lowest  0.7500  (seed 10)
highest 0.8667  (seed 12)
spread  0.1167
mean    0.8104   sd 0.0306
```

**Seed 12 is the highest of the twenty.** The report quoted 0.8667; the average across all
twenty is **0.8104**. The 11.7-point spread came from nothing but which 120 appointments
landed in the held-back pile.

**How to report it instead:** *"5-fold cross-validated accuracy 0.805, standard deviation
0.036."* That is a measurement. 0.8667 is a lottery ticket.

Note that nobody has to be dishonest for this to happen. You try a seed, the number looks
low, you wonder if it was unlucky, you try another — and you stop as soon as one looks
good. The stopping rule is the problem.

### B3 · The folds that were never shuffled

In [11]:
# the clinic's export tool writes the no-shows at the end of the file
exported = clinic.sort_values("no_show").reset_index(drop=True)
Xe = pd.get_dummies(exported[NUM + CAT], columns=CAT, drop_first=True).astype(float)
Xe["distance_km"] = Xe["distance_km"].fillna(Xe["distance_km"].median())
ye = exported.no_show

scores = cross_val_score(LogisticRegression(max_iter=2000), Xe, ye, cv=KFold(5))
print("fold scores:", np.round(scores, 4))
print(f"reported   : {scores.mean():.4f}")

fold scores: [0.775  0.8625 0.7625 0.5375 0.5375]
reported   : 0.6950


**The problem:** `KFold(5)` with no `shuffle` takes five **consecutive blocks** of a file
that is sorted by the outcome.

**The fix:** `StratifiedKFold(5, shuffle=True, random_state=0)`.

In [12]:
for name, cvo in [("KFold(5), no shuffle",         KFold(5)),
                  ("KFold(5), shuffle=True",       KFold(5, shuffle=True, random_state=0)),
                  ("StratifiedKFold(5), shuffled", StratifiedKFold(5, shuffle=True, random_state=0))]:
    rates = [ye.values[te].mean() for _, te in cvo.split(Xe, ye)]
    sc = cross_val_score(LogisticRegression(max_iter=2000), Xe, ye, cv=cvo)
    print(f"{name}")
    print(f"   no-show rate inside each fold : {np.round(rates, 4)}")
    print(f"   accuracy on each fold         : {np.round(sc, 4)}")
    print(f"   reported                      : {sc.mean():.4f}")
    print()

KFold(5), no shuffle
   no-show rate inside each fold : [0.    0.    0.    0.975 1.   ]
   accuracy on each fold         : [0.775  0.8625 0.7625 0.5375 0.5375]
   reported                      : 0.6950

KFold(5), shuffle=True
   no-show rate inside each fold : [0.375  0.4125 0.3375 0.3875 0.4625]
   accuracy on each fold         : [0.825  0.8125 0.8125 0.7875 0.775 ]
   reported                      : 0.8025



StratifiedKFold(5), shuffled
   no-show rate inside each fold : [0.3875 0.3875 0.4    0.4    0.4   ]
   accuracy on each fold         : [0.75   0.8375 0.7875 0.8    0.8125]
   reported                      : 0.7975



```
KFold(5), no shuffle
   no-show rate inside each fold : [0.    0.    0.    0.975 1.   ]
   accuracy on each fold         : [0.775  0.8625 0.7625 0.5375 0.5375]
   reported                      : 0.6950

KFold(5), shuffle=True
   no-show rate inside each fold : [0.375  0.4125 0.3375 0.3875 0.4625]
   accuracy on each fold         : [0.825  0.8125 0.8125 0.7875 0.775 ]
   reported                      : 0.8025

StratifiedKFold(5), shuffled
   no-show rate inside each fold : [0.3875 0.3875 0.4    0.4    0.4   ]
   accuracy on each fold         : [0.75   0.8375 0.7875 0.8    0.8125]
   reported                      : 0.7975
```

**Folds 1, 2 and 3 contain no no-shows at all.** Folds 4 and 5 are almost nothing but.

So the model is trained three times on a set containing every no-show in the dataset,
then tested on patients who all turned up — and twice on a set containing almost none,
then tested on patients who nearly all failed to. Folds 4 and 5 score **0.5375**, barely
better than a coin.

**The reported number drops from 0.8025 to 0.6950 for a reason that has nothing to do
with the model.** One keyword argument.

### An honest detail worth noticing

`shuffle=True` alone gets you to **0.8025**. Adding `stratify` gives **0.7975** — very
slightly *lower*.

That is not a failure. Stratifying fixes the **class mix** (every fold is within 0.0125 of
the overall 0.395 rate, against 0.3375–0.4625 for plain shuffling); it does not promise a
higher score. Use both anyway: you want the fold composition to be right by construction,
not right by luck.

### B4 · The preparation that happened too early

In [13]:
analyst = clinic.copy()

#  the analyst's "cleaning" step, run on the whole file before any splitting
analyst["distance_km"] = analyst["distance_km"].fillna(analyst["distance_km"].median())
Xa = pd.get_dummies(analyst[NUM + CAT], columns=CAT, drop_first=True).astype(float)
Xa[NUM] = StandardScaler().fit_transform(Xa[NUM])

sc = cross_val_score(LogisticRegression(max_iter=2000), Xa, analyst.no_show,
                     cv=StratifiedKFold(5, shuffle=True, random_state=0))
print(f"cross-validated accuracy: {sc.mean():.4f}")

cross-validated accuracy: 0.8050


**The problem:** the median and the scaling were computed from **all 400 rows**, including
the rows each fold is about to be scored on. The preparation happened before the folds
existed.

**The fix:** put every step inside a `Pipeline`, so each one is refitted inside each fold.

**But before you assume the number is badly wrong — measure it.**

In [14]:
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), NUM),
    ("cat", OneHotEncoder(drop="first"), CAT)])
proper = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])

cv = StratifiedKFold(5, shuffle=True, random_state=0)
right = cross_val_score(proper, clinic[NUM + CAT], clinic.no_show, cv=cv)

print(f"prepared up front   : {sc.mean():.4f}")
print(f"inside the Pipeline : {right.mean():.4f}")
print(f"difference          : {sc.mean() - right.mean():+.4f}")

prepared up front   : 0.8050
inside the Pipeline : 0.8050
difference          : +0.0000


```
prepared up front   : 0.8050
inside the Pipeline : 0.8050
difference          : +0.0000
```

### The honest answer: the number was not wrong at all

This is a real mistake and you should still fix it — but **it cost exactly nothing here**,
and you should know that rather than believe a scare story.

Why: neither step looks at `y`. A median and a standard deviation barely move when you add
20% more rows, so there was almost nothing to leak.

**Compare with the version that does look at `y`.**

In [15]:
# add some columns the clinic also happened to record, which mean nothing
noise_rng = np.random.default_rng(3651)
junk = pd.DataFrame(noise_rng.normal(0, 1, (len(clinic), 300)),
                    columns=[f"j{i}" for i in range(300)])
wide = pd.concat([Xa.reset_index(drop=True), junk], axis=1)

picked = SelectKBest(f_classif, k=12).fit(wide, clinic.no_show)
leaky = cross_val_score(LogisticRegression(max_iter=2000),
                        picked.transform(wide), clinic.no_show, cv=cv)
honest = cross_val_score(Pipeline([("sel", SelectKBest(f_classif, k=12)),
                                   ("lr", LogisticRegression(max_iter=2000))]),
                         wide, clinic.no_show, cv=cv)
print(f"columns offered to the selector: {wide.shape[1]}")
print(f"select on everything, then CV  : {leaky.mean():.4f}")
print(f"selection inside the Pipeline  : {honest.mean():.4f}")
print(f"the leak                       : {leaky.mean() - honest.mean():+.4f}")

columns offered to the selector: 306
select on everything, then CV  : 0.8200
selection inside the Pipeline  : 0.7625
the leak                       : +0.0575


```
columns offered to the selector: 306
select on everything, then CV  : 0.8200
selection inside the Pipeline  : 0.7625
the leak                       : +0.0575
```

Add 300 meaningless columns and let `SelectKBest` choose from all of them using the
answers, and the score jumps **5.75 points above the truth** — on columns that are pure
random numbers.

Notice also that the honest version (0.7625) is *worse* than the clean 6-column pipeline
from earlier (0.8050). Throwing 300 junk columns at the problem genuinely hurt the model;
the leak then papered over the damage and made it look like an improvement.

### So what is the lesson?

Not "everything leaks". The lesson is:

> **The step that chooses using `y` is the dangerous one, and the danger grows with how
> many things it had to choose between.**

And the practical response is not to memorise which steps are safe. It is to put all of
them in a Pipeline so you never have to decide.

---
# Part C — Write the code

Five tasks on the clinic data.

### C1 · Baseline first

Before any model: what fraction of appointments were no-shows, and what is the accuracy
of the laziest possible model? Also break the no-show rate down by `reminder`.

In [16]:
# your code here

In [17]:
print(clinic.no_show.value_counts().to_string())
print(f"\nno-show rate: {clinic.no_show.mean():.4f}")
print(f"baseline ('everybody turns up'): {1 - clinic.no_show.mean():.4f}")
print()
print(clinic.groupby("reminder").no_show.agg(["size", "mean"]).round(4).to_string())

no_show
0    242
1    158

no-show rate: 0.3950
baseline ('everybody turns up'): 0.6050

          size    mean
reminder              
call        82  0.1951
none       135  0.5926
sms        183  0.3388


```
no_show
0    242
1    158

no-show rate: 0.3950
baseline ('everybody turns up'): 0.6050

          size    mean
reminder
call        82  0.1951
none       135  0.5926
sms        183  0.3388
```

**The baseline is 0.6050.** Any accuracy we report has to be read against that.

And before fitting anything, the reminder column already tells a story: **patients who got
a phone call missed 19.5% of appointments; patients who got nothing missed 59.3%.** Three
times the rate. We should expect the model to find that, and if it does not, something is
wrong with our code rather than with the world.

### C2 · Cross-validate properly

Build a `ColumnTransformer` + `LogisticRegression` pipeline that fills the missing
distances with a median, scales the numbers, and one-hot encodes `reminder`.

Cross-validate it with 5 stratified shuffled folds. Report the five fold scores, the mean,
the standard deviation, **and** the training score, so you can see the gap.

In [18]:
# your code here

In [19]:
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), NUM),
    ("cat", OneHotEncoder(drop="first"), CAT)])

full = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])

cv = StratifiedKFold(5, shuffle=True, random_state=0)
res = cross_validate(full, clinic[NUM + CAT], clinic.no_show,
                     cv=cv, return_train_score=True)

print("held-back fold scores:", np.round(res["test_score"], 4))
print("studied  fold scores :", np.round(res["train_score"], 4))
print()
print(f"held back  {res['test_score'].mean():.4f}   sd {res['test_score'].std(ddof=1):.4f}")
print(f"studied    {res['train_score'].mean():.4f}")
print(f"the gap    {res['train_score'].mean() - res['test_score'].mean():+.4f}")
print(f"baseline   {1 - clinic.no_show.mean():.4f}")

held-back fold scores: [0.85   0.75   0.8125 0.8    0.8125]
studied  fold scores : [0.8031 0.8281 0.8094 0.8156 0.8125]

held back  0.8050   sd 0.0360
studied    0.8137
the gap    +0.0087
baseline   0.6050


```
held-back fold scores: [0.85   0.75   0.8125 0.8    0.8125]
studied  fold scores : [0.8031 0.8281 0.8094 0.8156 0.8125]

held back  0.8050   sd 0.0360
studied    0.8137
the gap    +0.0087
baseline   0.6050
```

**Report it as: "0.805, give or take about 0.036."**

Three things worth saying:

- **0.8050 against a baseline of 0.6050** — a gain of 20 points. The model is doing real work.
- **The gap is 0.0087.** Studied and held-back agree, so this is not overfitting.
- **The spread is 0.036**, and the folds run from 0.75 to 0.85. On 80 rows per fold that is
  ordinary — one fold is 8 appointments away from another.

One honest note on that first fold: 0.85 is the best of the five. If you had run a single
split and happened to draw those rows, you would be reporting 0.85 right now.

### C3 · How many folds?

Run the same pipeline with 3, 5 and 10 folds. Report each mean and standard deviation, and
say which you would use here and why.

In [20]:
# your code here

In [21]:
print(f"{'':>10}{'mean':>10}{'sd':>10}{'fits':>7}{'rows per fold':>16}")
print("-" * 53)
for label, k in [("3-fold", 3), ("5-fold", 5), ("10-fold", 10)]:
    s_ = cross_val_score(full, clinic[NUM + CAT], clinic.no_show,
                         cv=StratifiedKFold(k, shuffle=True, random_state=0))
    print(f"{label:>10}{s_.mean():>10.4f}{s_.std(ddof=1):>10.4f}{k:>7}{len(clinic)/k:>16.0f}")

                mean        sd   fits   rows per fold
-----------------------------------------------------
    3-fold    0.8075    0.0221      3             133


    5-fold    0.8050    0.0360      5              80


   10-fold    0.7975    0.0506     10              40


```
                mean        sd   fits   rows per fold
-----------------------------------------------------
    3-fold    0.8075    0.0221      3             133
    5-fold    0.8050    0.0360      5              80
   10-fold    0.7975    0.0506     10              40
```

**The means barely move** — 0.8075, 0.8050, 0.7975, all within 1 point. That is reassuring:
the estimate is not sensitive to how you slice it.

**The standard deviations more than double**, 0.0221 → 0.0506, and that is not the model
becoming less stable. It is each fold being scored on fewer rows: 133, then 80, then 40.
With 40 rows a single appointment is worth 2.5 points.

**Use 5 here.** 3 folds trains on only two thirds of the data; 10 gives fold scores too
noisy to interpret on 400 rows. If this were 50,000 rows, 10 would be fine and the
standard deviations would all be tiny.

**A caution about reading that table:** do not pick the fold count that gives the nicest
number. Choose it from the size of the data, before you look.

### C4 · Fit it on everything and read the coefficients

Fit the pipeline on all 400 rows. Print each column the pipeline built, its coefficient,
and what it does to the odds. Which column matters most, and what should the clinic
actually do?

In [22]:
# your code here

In [23]:
full.fit(clinic[NUM + CAT], clinic.no_show)

names = full.named_steps["prep"].get_feature_names_out()
coefs = full.named_steps["model"].coef_[0]
order = np.argsort(-np.abs(coefs))

for i in order:
    print(f"   {names[i]:<32}{coefs[i]:>+8.4f}   odds x {np.exp(coefs[i]):.3f}")
print(f"   {'intercept':<32}{full.named_steps['model'].intercept_[0]:>+8.4f}")
print()
print("median distance used for the gaps:",
      full.named_steps["prep"].named_transformers_["num"].named_steps["imp"].statistics_[3])

   cat__reminder_none               +2.0989   odds x 8.157
   num__previous_no_shows           +1.3788   odds x 3.970
   num__lead_time_days              +1.1323   odds x 3.103
   cat__reminder_sms                +0.6661   odds x 1.947
   num__age                         -0.6371   odds x 0.529
   num__distance_km                 +0.2584   odds x 1.295
   intercept                        -1.6588

median distance used for the gaps: 8.8


```
   cat__reminder_none               +2.0989   odds x 8.157
   num__previous_no_shows           +1.3788   odds x 3.970
   num__lead_time_days              +1.1323   odds x 3.103
   cat__reminder_sms                +0.6661   odds x 1.947
   num__age                         -0.6371   odds x 0.529
   num__distance_km                 +0.2584   odds x 1.295
   intercept                        -1.6588

median distance used for the gaps: 8.8
```

**`reminder_none` dominates: ×8.16 on the odds.** The dropped baseline is `call`, so this
says that sending nothing instead of phoning multiplies the odds of a no-show by more
than eight. An SMS is better than nothing (×1.95) but far worse than a call.

**Because the numbers were scaled, these coefficients are comparable** — that is Day 34's
lesson arriving for free from the pipeline. One standard deviation more history of
no-shows (×3.97) or of lead time (×3.10) each matter more than anything except the
reminder.

### What the clinic should actually do

**Phone people.** It is the largest effect in the model and the only one the clinic
controls. Age, distance and previous history are facts about the patient; the reminder
method is a decision made every day.

A second, cheaper action: **shorten lead times**. Booking people 50 days out instead of 10
roughly triples the no-show odds, and appointment scheduling is also within the clinic's
control.

> **An honest caveat.** This is observational data, not an experiment. Maybe the clinic
> already phones the patients it judges most likely to miss, in which case the real effect
> of phoning is *larger* than it looks here. Or maybe phone numbers are only on file for
> more engaged patients, in which case it is smaller. A coefficient is not a causal
> effect, and the only way to settle this is to assign reminders at random for a month.

### C5 · Save it and use it on three new patients

Fit the pipeline, save it with `joblib`, load it back, and score these three:

| | age | lead time | previous no-shows | distance | reminder |
|---|---|---|---|---|---|
| **A** | 23 | 52 days | 3 | 22 km | none |
| **B** | 61 | 4 days | 0 | 3 km | call |
| **C** | 35 | 21 days | 1 | **unknown** | sms |

Patient C's address is missing — make sure that does not break anything.

In [24]:
# your code here

In [25]:
import joblib

full.fit(clinic[NUM + CAT], clinic.no_show)
joblib.dump(full, "clinic_model.joblib")
loaded = joblib.load("clinic_model.joblib")

new = pd.DataFrame([
    {"age": 23, "lead_time_days": 52, "previous_no_shows": 3,
     "distance_km": 22.0, "reminder": "none"},
    {"age": 61, "lead_time_days": 4,  "previous_no_shows": 0,
     "distance_km": 3.0,  "reminder": "call"},
    {"age": 35, "lead_time_days": 21, "previous_no_shows": 1,
     "distance_km": np.nan, "reminder": "sms"},
])

p = loaded.predict_proba(new[NUM + CAT])[:, 1]
for label, prob in zip("ABC", p):
    print(f"patient {label}:  p(no-show) = {prob:.4f}")

patient A:  p(no-show) = 0.9982
patient B:  p(no-show) = 0.0017
patient C:  p(no-show) = 0.1848


```
patient A:  p(no-show) = 0.9982
patient B:  p(no-show) = 0.0017
patient C:  p(no-show) = 0.1848
```

**Patient A — 99.8%.** Young, booked seven weeks out, three previous misses, 22 km away,
and nobody is going to call. Every single factor points the same way, and they compound.
Phone this patient.

**Patient B — 0.17%.** Older, booked for next week, never missed before, lives nearby, and
is getting a call. Essentially certain to turn up.

**Patient C — 18.5%, and the missing address did not break anything.** The pipeline filled
it with the median distance (8.8 km) it learned during training, scaled it with the mean
and standard deviation it learned during training, and encoded `sms` against the same
baseline. **None of that information exists outside the saved file.**

> **A word of caution about A and B.** 0.9982 and 0.0017 are more confident than 400 rows
> can really justify — these two patients sit at opposite extremes of every column at
> once, and logistic regression extrapolates cheerfully. Treat them as "very likely" and
> "very unlikely", not as four-figure precision. That is Day 34's warning about predicting
> outside the range you have seen.

### Why this task was the point of the whole day

Save the `LogisticRegression` on its own and you would have to remember, by hand and
correctly, the median distance, four means, four standard deviations, the category list
and the column order. Get any one wrong and the model keeps working and quietly returns
rubbish.

> **A model on its own is not deployable. A pipeline is.**

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Write k-fold cross-validation yourself

Write `my_cross_val_score(model, X, y, k=5, seed=0)` using **no** scikit-learn
cross-validation tools. You may use `clone` from `sklearn.base`.

It should:

- shuffle the row numbers, then cut them into `k` groups as evenly as possible
  (with 400 rows and k=3, the groups must be 134/133/133 — not 133/133/134 with one lost)
- for each group: fit a **fresh** copy of the model on the other rows, score it on that group
- return the `k` scores as an array

Then prove it matches `cross_val_score` with `KFold(k, shuffle=True, random_state=seed)`.

**The hard part is matching scikit-learn's grouping exactly.** Read `KFold`'s rule: with
`n` rows and `k` folds, the first `n % k` folds get `n//k + 1` rows and the rest get
`n//k`.

In [26]:
demo_X = flat.values
demo_y = target.values
print(demo_X.shape, demo_y.shape)

(400, 6) (400,)


In [27]:
# your function here

In [28]:
from sklearn.base import clone

def my_cross_val_score(model, X, y, k=5, seed=0):
    """k-fold cross-validation, written out. Matches KFold(shuffle=True)."""
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(y)

    # 1 · shuffle the row numbers exactly as sklearn does
    order = np.arange(n)
    np.random.RandomState(seed).shuffle(order)

    # 2 · cut them into k groups: the first (n % k) groups get one extra row
    sizes = np.full(k, n // k)
    sizes[: n % k] += 1
    groups, start = [], 0
    for s in sizes:
        groups.append(order[start:start + s])
        start += s

    # 3 · each group takes one turn as the held-back set
    scores = []
    for g in groups:
        held = np.zeros(n, dtype=bool)
        held[g] = True
        fresh = clone(model)                 # a brand new model every round
        fresh.fit(X[~held], y[~held])
        scores.append(fresh.score(X[held], y[held]))
    return np.array(scores)

In [29]:
mine = my_cross_val_score(LogisticRegression(max_iter=2000), demo_X, demo_y, k=5, seed=0)
theirs = cross_val_score(LogisticRegression(max_iter=2000), demo_X, demo_y,
                         cv=KFold(5, shuffle=True, random_state=0))

print("mine  :", np.round(mine, 6))
print("theirs:", np.round(theirs, 6))
print("identical:", np.allclose(mine, theirs))
print(f"\nmine   mean {mine.mean():.6f}  sd {mine.std(ddof=1):.6f}")
print(f"theirs mean {theirs.mean():.6f}  sd {theirs.std(ddof=1):.6f}")

mine  : [0.8125 0.8    0.8375 0.8    0.7875]
theirs: [0.8125 0.8    0.8375 0.8    0.7875]
identical: True

mine   mean 0.807500  sd 0.018957
theirs mean 0.807500  sd 0.018957


In [30]:
# prove the group sizes are right, including when k does not divide n
for n_, k_ in [(400, 5), (400, 3), (400, 7), (101, 4), (10, 3)]:
    sizes = np.full(k_, n_ // k_); sizes[: n_ % k_] += 1
    print(f"n={n_:>4} k={k_}: groups {sizes.tolist()}  sum {sizes.sum()}  "
          f"max-min {sizes.max()-sizes.min()}")

n= 400 k=5: groups [80, 80, 80, 80, 80]  sum 400  max-min 0
n= 400 k=3: groups [134, 133, 133]  sum 400  max-min 1
n= 400 k=7: groups [58, 57, 57, 57, 57, 57, 57]  sum 400  max-min 1
n= 101 k=4: groups [26, 25, 25, 25]  sum 101  max-min 1
n=  10 k=3: groups [4, 3, 3]  sum 10  max-min 1


In [31]:
# and that it works for other models and other k
for k_ in [3, 5, 10]:
    for nm, m in [("LogisticRegression", LogisticRegression(max_iter=2000)),
                  ("DecisionTree d=3", DecisionTreeClassifier(max_depth=3, random_state=0)),
                  ("KNN k=15", KNeighborsClassifier(15))]:
        a = my_cross_val_score(m, demo_X, demo_y, k=k_, seed=1)
        b = cross_val_score(m, demo_X, demo_y, cv=KFold(k_, shuffle=True, random_state=1))
        print(f"k={k_:>2}  {nm:<20} match {np.allclose(a, b)}   mean {a.mean():.4f}")

k= 3  LogisticRegression   match True   mean 0.8124


k= 3  DecisionTree d=3     match True   mean 0.7000


k= 3  KNN k=15             match True   mean 0.6601


k= 5  LogisticRegression   match True   mean 0.7950
k= 5  DecisionTree d=3     match True   mean 0.7300


k= 5  KNN k=15             match True   mean 0.6675


k=10  LogisticRegression   match True   mean 0.8000


k=10  DecisionTree d=3     match True   mean 0.7425


k=10  KNN k=15             match True   mean 0.6700


### The result

```
mine  : [0.8125 0.8    0.8375 0.8    0.7875]
theirs: [0.8125 0.8    0.8375 0.8    0.7875]
identical: True

mine   mean 0.807500  sd 0.018957
theirs mean 0.807500  sd 0.018957
```

Group sizes, including the awkward cases:

```
n= 400 k=5: groups [80, 80, 80, 80, 80]  sum 400  max-min 0
n= 400 k=3: groups [134, 133, 133]  sum 400  max-min 1
n= 400 k=7: groups [58, 57, 57, 57, 57, 57, 57]  sum 400  max-min 1
n= 101 k=4: groups [26, 25, 25, 25]  sum 101  max-min 1
n=  10 k=3: groups [4, 3, 3]  sum 10  max-min 1
```

And it is not a fluke of one model:

```
k= 3  LogisticRegression   match True   mean 0.8124
k= 3  DecisionTree d=3     match True   mean 0.7000
k= 3  KNN k=15             match True   mean 0.6601
k= 5  LogisticRegression   match True   mean 0.7950
k= 5  DecisionTree d=3     match True   mean 0.7300
k= 5  KNN k=15             match True   mean 0.6675
k=10  LogisticRegression   match True   mean 0.8000
k=10  DecisionTree d=3     match True   mean 0.7425
k=10  KNN k=15             match True   mean 0.6700
```

(KNN scores badly here because `demo_X` is unscaled — distances are dominated by
`lead_time_days`, which runs to 60. That is Day 32, and it is not what this question is
testing, but it is worth seeing.)

### The four things that earn the marks

**1 · `clone(model)` inside the loop.** Without it you refit the *same object* five times,
and while scikit-learn's `.fit` does reset most estimators, relying on that is how subtle
bugs happen. A fold must get a model that has never seen anything.

**2 · The remainder handling.** `np.array_split`-style thinking gets this right, but
writing `n // k` and dropping the remainder loses up to `k-1` rows silently — the scores
would still look perfectly reasonable. The first `n % k` groups take one extra row each.

**3 · Shuffling the indices, not the data.** Shuffle row *numbers* and index into `X` and
`y` with them. Shuffling two arrays separately is a classic way to decouple rows from
their labels, and the only symptom is a score near chance.

**4 · `np.random.RandomState(seed).shuffle`** is what scikit-learn uses internally, which
is why the match is exact rather than merely close. The modern `default_rng` shuffles
differently — equally valid, but it would not reproduce their folds.

### What this exercise is really for

There is no magic in `cross_val_score`. It is fifteen lines: shuffle, cut, loop, fit,
score. Once you have written it, "the transformer is refitted inside every fold" stops
being something you were told and becomes something you can see — it is just where in
this loop you put the `.fit`.

### D2 · Audit a model the company is already using

A company has deployed a model that predicts which employees will leave within a year.
HR uses its output to decide who gets a retention conversation. The analyst's report says:

> *"Cross-validated accuracy **0.906** on 2,000 employee records, using 44 features. The
> model is in production and performing well."*

The code below is the analyst's, reproduced exactly. It runs without error and the 0.906
is real.

**Your tasks:**

1. Run it and confirm the 0.906.
2. Find **three separate things wrong** with it.
3. Prove each with numbers.
4. Write a recommendation for the Head of HR — not a lecture on methodology, a decision.

In [32]:
r = np.random.default_rng(3660)
N = 2000

department   = r.choice(["sales", "eng", "support", "ops"], N, p=[.28, .34, .22, .16])
tenure_years = r.uniform(0.2, 15, N).round(1)
salary_band  = r.integers(1, 8, N).astype(float)
commute_min  = r.normal(18, 11, N).clip(1, 80).round(0)
last_rating  = r.normal(3.4, 0.8, N).clip(1, 5).round(1)

db = np.where(department == "sales", 0.6,
     np.where(department == "support", 0.35,
     np.where(department == "eng", -0.2, 0.0)))
z = (0.70 - 0.16*tenure_years - 0.22*salary_band + 0.020*commute_min
     - 0.55*last_rating + db + r.normal(0, 0.4, N))
left = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

hr = pd.DataFrame({"tenure_years": tenure_years, "salary_band": salary_band,
                   "commute_minutes": commute_min, "last_rating": last_rating,
                   "department": department})
#  40 engagement-survey questions the analyst also had available
survey = r.normal(0, 1, (N, 40))
for i in range(40):
    hr[f"survey_q{i+1}"] = survey[:, i].round(3)
hr["left_company"] = left

#  the HR system exports records grouped by department
hr = hr.sort_values("department").reset_index(drop=True)
hr.head()

,tenure_years,salary_band,commute_minutes,last_rating,department,survey_q1,survey_q2,survey_q3,survey_q4,survey_q5,...,survey_q32,survey_q33,survey_q34,survey_q35,survey_q36,survey_q37,survey_q38,survey_q39,survey_q40,left_company
0,5.9,4.0,28.0,3.5,eng,0.707,0.143,0.880,-0.197,-2.562,...,0.925,1.401,0.629,0.329,-0.898,-0.971,0.238,-0.419,2.013,0
1,2.6,7.0,6.0,2.8,eng,0.274,0.992,0.081,-1.872,1.530,...,-0.593,1.211,-1.323,-0.977,0.126,0.101,-0.912,-1.280,-0.879,0
2,4.3,2.0,7.0,3.1,eng,0.372,2.012,0.841,1.384,-0.030,...,-0.158,-0.536,-0.021,0.935,0.771,-0.464,3.125,1.143,-1.263,0
3,3.9,4.0,36.0,4.6,eng,0.775,1.222,0.929,0.812,-0.016,...,0.298,2.100,0.437,0.491,-0.057,1.384,-0.820,-0.632,0.626,0
4,8.5,4.0,1.0,5.0,eng,-0.194,-0.818,1.184,1.535,0.279,...,0.439,0.473,0.433,-1.533,-0.835,-0.104,0.313,1.477,-0.945,0


In [33]:
#  ---- the analyst's code, exactly as written ----
features = [c for c in hr.columns if c not in ("left_company", "department")]
Xh = hr[features].values
yh = hr.left_company.values

Xh_scaled = StandardScaler().fit_transform(Xh)
best10 = SelectKBest(f_classif, k=10).fit(Xh_scaled, yh)

report = cross_val_score(LogisticRegression(max_iter=2000),
                         best10.transform(Xh_scaled), yh, cv=KFold(5))

print(f"cross-validated accuracy: {report.mean():.4f}")
print(f"fold scores             : {np.round(report, 4)}")

cross-validated accuracy: 0.9060
fold scores             : [0.94   0.9525 0.88   0.8425 0.915 ]


In [34]:
# your investigation here

In [35]:
# ---- PROBLEM 1 : what does doing nothing score? -----------------------
print("how many people actually left?")
print(f"   leave rate            : {yh.mean():.4f}")
print(f"   'nobody ever leaves'  : {1 - yh.mean():.4f}")
print(f"   the analyst's model   : {report.mean():.4f}")
print(f"   the gain              : {report.mean() - (1 - yh.mean()):+.4f}")

how many people actually left?
   leave rate            : 0.0935
   'nobody ever leaves'  : 0.9065
   the analyst's model   : 0.9060
   the gain              : -0.0005


In [36]:
# ---- PROBLEM 2 : what does it actually catch? -------------------------
proper_cv = StratifiedKFold(5, shuffle=True, random_state=0)
safe = Pipeline([("sc", StandardScaler()),
                 ("sel", SelectKBest(f_classif, k=10)),
                 ("lr", LogisticRegression(max_iter=2000))])

pred = cross_val_predict(safe, Xh, yh, cv=proper_cv)
tn, fp, fn, tp = confusion_matrix(yh, pred).ravel()
print(f"of the {int(yh.sum())} people who left:")
print(f"   the model flagged : {tp}")
print(f"   the model missed  : {fn}")
print(f"   recall            : {tp/(tp+fn):.4f}")
print(f"\nit flagged {tp+fp} people in total, of whom {fp} did not leave")

of the 187 people who left:
   the model flagged : 2
   the model missed  : 185
   recall            : 0.0107

it flagged 4 people in total, of whom 2 did not leave


In [37]:
# ---- PROBLEM 3 : the folds, on a file sorted by department ------------
rates = [yh[te].mean() for _, te in KFold(5).split(Xh, yh)]
print("leave rate inside each KFold(5) fold:", np.round(rates, 4))
print("the analyst's fold scores           :", np.round(report, 4))
print(f"   those folds range over {report.max()-report.min():.4f}")
print()
depts = [hr.department.values[te] for _, te in KFold(5).split(Xh, yh)]
for i, d in enumerate(depts):
    print(f"   fold {i+1} departments: {sorted(set(d))}")

leave rate inside each KFold(5) fold: [0.055  0.05   0.1175 0.1575 0.0875]
the analyst's fold scores           : [0.94   0.9525 0.88   0.8425 0.915 ]
   those folds range over 0.1100

   fold 1 departments: ['eng']
   fold 2 departments: ['eng', 'ops']
   fold 3 departments: ['ops', 'sales']
   fold 4 departments: ['sales', 'support']
   fold 5 departments: ['support']


In [38]:
# ---- and the leak the analyst also has, measured ----------------------
leak_free = cross_val_score(safe, Xh, yh, cv=KFold(5))
both_fixed = cross_val_score(safe, Xh, yh, cv=proper_cv)
print(f"the analyst's number            : {report.mean():.4f}")
print(f"fixing the leak only            : {leak_free.mean():.4f}")
print(f"fixing the leak AND the folds   : {both_fixed.mean():.4f}")
print(f"'nobody ever leaves'            : {1 - yh.mean():.4f}")
print()
real = ["tenure_years", "salary_band", "commute_minutes", "last_rating"]
only_real = cross_val_score(
    Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))]),
    hr[real].values, yh, cv=proper_cv)
print(f"the 4 real columns, done properly: {only_real.mean():.4f}")

the analyst's number            : 0.9060
fixing the leak only            : 0.9055
fixing the leak AND the folds   : 0.9065
'nobody ever leaves'            : 0.9065

the 4 real columns, done properly: 0.9055


### Problem 1 — 0.906 is the baseline

```
   leave rate            : 0.0935
   'nobody ever leaves'  : 0.9065
   the analyst's model   : 0.9060
   the gain              : -0.0005
```

**Only 9.35% of employees left**, so a system that predicts *nobody ever leaves* — no
model, no features, no survey — scores **0.9065**.

The analyst's model scores **0.9060**. That is not a small gain. **It is slightly worse
than doing nothing at all.**

Every other problem below is secondary to this one. The headline number was never
evidence of anything, and one line of code would have revealed it.

### Problem 2 — it flags almost nobody

```
of the 187 people who left:
   the model flagged : 2
   the model missed  : 185
   recall            : 0.0107

it flagged 4 people in total, of whom 2 did not leave
```

**The model identified two people.** HR is using it to decide who gets a retention
conversation, and across two thousand employees it nominated four names, half of them
wrong.

This is what high accuracy on a rare outcome always means: the model has discovered that
saying "no" is almost always right, and that is all it has discovered.

### Problem 3 — the folds were built from a file sorted by department

```
leave rate inside each KFold(5) fold: [0.055  0.05   0.1175 0.1575 0.0875]
the analyst's fold scores           : [0.94   0.9525 0.88   0.8425 0.915 ]
   those folds range over 0.1100

   fold 1 departments: ['eng']
   fold 2 departments: ['eng', 'ops']
   fold 3 departments: ['ops', 'sales']
   fold 4 departments: ['sales', 'support']
   fold 5 departments: ['support']
```

The HR system exports records grouped by department, and `KFold(5)` with no `shuffle`
took five consecutive blocks. **Fold 1 is nothing but engineering. Fold 5 is nothing but
support.** No fold contains a representative mix of the company.

Since departments have very different leave rates, the fold leave rates range from **0.050
to 0.1575** — a factor of three — and the fold accuracies swing **11 points** for that
reason alone. The reported 0.906 is an average over five measurements of five different
populations.

### The leak, measured — and why it is the least of it

```
the analyst's number            : 0.9060
fixing the leak only            : 0.9055
fixing the leak AND the folds   : 0.9065
'nobody ever leaves'            : 0.9065
the 4 real columns, done properly: 0.9055
```

The analyst also scaled and ran `SelectKBest` across all 2,000 rows before
cross-validating — a genuine leak, and `SelectKBest` is the dangerous kind that looks at
`y`.

**And fixing it changed the score by 0.0005.**

That is worth sitting with. The textbook mistake was present, and it was irrelevant,
because there was never any signal for it to inflate. Fixing everything lands on 0.9065 —
**exactly the baseline**, to four decimal places.

### The recommendation

> **Stop using this model for retention decisions.**
>
> Its 90.6% accuracy is not a result: 90.6% of our employees stay, so predicting that
> nobody ever leaves scores the same. Across 2,000 records the model nominated **four**
> people for a retention conversation, and correctly identified **two of the 187** who
> actually left. HR is receiving almost no information from it.
>
> The underlying problem is not a coding error — I found and fixed two, and neither moved
> the number. It is that the 44 features we collected, including all 40 survey questions,
> do not predict leaving well enough to be useful at this base rate.
>
> **What I would do instead.** Ask the model for a *ranked list* rather than a yes/no, and
> have HR work down the top 50 — that uses whatever weak signal exists without pretending
> to certainty. In parallel, gather features that plausibly carry more signal: time since
> last promotion, manager changes, pay relative to market, internal application activity.
> And from now on, report every accuracy next to the do-nothing baseline, which would have
> caught this in week one.

### The lesson

The leak was the mistake you were trained to look for, and it was the least important
thing wrong here. **Check the baseline first.** It costs one line, and it is the only
check that would have stopped this model reaching production.

---
## Before you hand it in

- Is every score you quote measured on rows the model **never saw**?
- Is every accuracy sitting **next to its baseline**?
- Did you report a **mean and a spread**, not a single number?
- Is `shuffle=True` on, and `stratify` where it belongs?
- Is every preparation step **inside** the Pipeline?
- If you compared several models, did you keep a **final test set** back?

Five numbers worth carrying away:

- A decision tree scored **1.0000** on the appointments it studied and **0.7250** on the
  ones it had not — the worst of the three models on the only column that counts.
- Twenty splits of the same 400 appointments ranged over **11.7 points**. The report
  quoted the highest.
- A file sorted by outcome dropped the reported accuracy from **0.8125 to 0.6825**, with
  three folds containing no no-shows at all.
- Scaling and median-filling before the split leaked **+0.0000**. Adding 300 junk columns
  and letting `SelectKBest` choose leaked **+0.0225**.
- A deployed HR model with **0.906 accuracy** matched its baseline exactly and caught
  **2 of 187** leavers.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*